# ACV — plateforme photovoltaïque flottante, lagon de Ra'iātea

Ce carnet reproduit, dans l'ordre de l'article, **tous les chiffres publiés** et
rien d'autre. Chaque section indique le § de l'article, la figure ou le tableau
qu'elle produit.

**C'est une copie figée.** Le dossier de travail reste `acv_lagon/` à la racine :
celui-ci ne doit servir qu'à reproduire, pas à expérimenter.

| | |
|---|---|
| Base de fond | ecoinvent 3.11, system model cut-off |
| Méthode | EF v3.1, 16 catégories midpoint |
| Lecture secondaire | ReCiPe 2016 v1.03 endpoint (H) |
| Unité fonctionnelle | 1 kWh net livré au réseau, sur 30 ans |
| Configuration de référence | S2 (SH51, 51 % opaque) |

**Durée.** Le carnet complet demande une trentaine de minutes : le Monte-Carlo à
500 000 tirages et les transpositions territoriales dominent. Les cellules
coûteuses portent un avertissement.

## 1. Initialisation

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pathlib
import sys

# ══════════════════════════════════════════════════════════════════════════
# LE PAQUET VIENT DE CE DOSSIER, ET DE NULLE PART AILLEURS
#
# Le dossier de travail porte les MÊMES noms de modules. Sans une source
# unique et explicite, le noyau peut charger l'un pour l'autre sans la
# moindre erreur. Les trois lignes ci-dessous l'interdisent.
#
# Le carnet vit DANS le dossier qui contient acv_lagon/, donc la racine est
# simplement son propre dossier. Si tu le lances d'ailleurs, remplace par le
# chemin en dur :  RACINE = pathlib.Path("~/ACV_VF").expanduser().resolve()
# ══════════════════════════════════════════════════════════════════════════
RACINE = pathlib.Path.cwd().resolve()

if not (RACINE / "acv_lagon" / "__init__.py").exists():
    raise FileNotFoundError(
        f"acv_lagon/ introuvable dans {RACINE}. Le carnet doit etre dans le "
        "meme dossier que acv_lagon/, ou RACINE doit pointer dessus "
        "explicitement.")

# Purge : un acv_lagon deja charge primerait sur sys.path, quoi qu'on fasse.
for _nom in [n for n in list(sys.modules)
             if n == "acv_lagon" or n.startswith("acv_lagon.")]:
    del sys.modules[_nom]

sys.path.insert(0, str(RACINE))

# Garde-fou du noyau : le projet Brightway est ecrit par Python 3.11
# (pickle protocole 5). Depuis un noyau plus ancien, l'import de brightway2
# echoue sur un message obscur dont la vraie cause est le choix du noyau.
print("Interpreteur :", sys.executable)
if sys.version_info < (3, 8):
    raise RuntimeError("Kernel -> Change kernel -> l'environnement du projet.")

from acv_lagon import (projet, systeme, modalites, resultats, checks, config,
                       fin_de_vie, parametres, structure, ecoinvent, tests,
                       territoire, litterature, incertitude, economie, endpoint)
import acv_lagon
import pandas as pd
import lca_algebraic as agb

# Controle : il vient bien de RACINE ?
_vient_de = pathlib.Path(acv_lagon.__file__).parent.parent.resolve()
assert _vient_de == RACINE, f"paquet charge depuis {_vient_de}, pas depuis {RACINE}"
print("Paquet       :", RACINE / "acv_lagon")

agb.resetParams()
ctx = projet.initialiser()
ctx

## 2. Construction du modèle

In [ ]:
systeme.construire(ctx)

## 3. Les quatre configurations

`checks` relit les paramètres et recalcule la masse de modules depuis les fiches
constructeur : une erreur de jeu de paramètres se voit ici, pas trois heures plus
tard dans une figure.

In [ ]:
jeux = modalites.jeux_de_parametres(ctx)
REF = "S2 (SH51, opaque 51%)"

for libelle, valeurs in jeux.items():
    checks.verifier_parametres(valeurs)
    checks.verifier_masse_panneaux(ctx, valeurs, libelle=libelle)

## 4. Résultat principal — § 3.1, tableau 2, figure 4

`reference` fige le GWP de la configuration de référence : toute cellule qui le
recalcule plus bas s'y compare, et un écart lève une erreur au lieu de passer
inaperçu.

In [ ]:
df = resultats.comparer_modalites(ctx, jeux)
reference = checks.Reference(df.loc[resultats._ligne_gwp(df), REF],
                             libelle=f"GWP {REF}")
df

### Sensibilité au taux de dégradation — tableau 3 et annexe S3

In [ ]:
resultats.sensibilite_degradation(ctx)   # 0,35 (référence) / 0,50 / 0,70 / 1,00 %/an

### Sensibilité à la disponibilité — 97 % (référence) contre 100 %, tableau 3

In [ ]:
resultats.sensibilite_disponibilite(ctx)   # 97 % (référence) / 100 % (productible reconstruit)

### Bande d'incertitude de la figure 4

In [ ]:
bande = incertitude.incertitude(ctx, jeux, n=5000, seed=0)
resultats.figure_gwp(df, incertitudes=bande);

In [ ]:
bande[["code", "nominal g/kWh", "p5", "p50", "p95",
       "demi-largeur %", "CV %", "écart nominal %"]].round(2)

### Effet de la durée de vie retenue

In [ ]:
resultats.plage_duree_de_vie(ctx, jeux).round(1)

## 5. Contribution par poste — § 3.2, figures 5 et 7, tableau 4

In [ ]:
contrib = resultats.contribution(ctx, jeux[REF], exacte=True)
(contrib[contrib.columns[0]] * 1000).round(2).sort_values(ascending=False)

In [ ]:
resultats.figure_schema_systeme(ctx, jeux[REF]);

In [ ]:
resultats.figure_contribution_postes(ctx, jeux, n_sous=1, ncols=2);

In [ ]:
resultats.figure_camembert(ctx, jeux[REF]);

In [ ]:
resultats.table_contribution_detaillee(ctx, jeux, n_sous=3).round(2)

### Les seize catégories, parts en pourcent — figure 7

In [ ]:
resultats.figure_barres_100(ctx=ctx, jeux=jeux);

In [ ]:
resultats.comparer_contributions(ctx, jeux[REF])

## 6. Incertitude et sensibilité — § 2.6 et § 3.2, annexe S3

Les lois déclarées pour chaque paramètre sont dans l'annexe S3 de l'article ;
`apercu_distributions()` en est la source.

In [ ]:
resultats.apercu_distributions().sort_values("demi-plage %", ascending=False)

In [ ]:
tornade = incertitude.tornado_modalites(ctx, jeux, n_parametres=10)
incertitude.figure_tornado_modalites(tornade);

In [ ]:
reference.verifier(systeme.gwp(ctx, **jeux[REF]), "nominal du tornado", strict=True)

In [ ]:
elas = resultats.matrice_elasticite(ctx, jeux[REF], variation=0.20)
elas.round(2)

⚠ 500 000 tirages par configuration : compter une dizaine de minutes.

In [ ]:
echs = incertitude.echantillons(ctx, jeux, n=500000, seed=0)
incertitude.figure_incertitude(echs, bande);

In [ ]:
variance = incertitude.variance_modalites(ctx, jeux, echs=echs, n_parametres=12)
incertitude.figure_variance_modalites(variance);

### Variante de maintenance — un SCÉNARIO, pas une incertitude

In [ ]:
maint = resultats.variante_maintenance(ctx, jeux, rythmes=(15, 26))
maint.round(3)

In [ ]:
resultats.figure_variante_maintenance(maint);

## 7. Fin de vie — § 2.5 et § 3.5, figure 11

Sept modalités : enfouissement local, recyclage NZ en cut-off, recyclage NZ en
closed-loop, puis quatre variantes d'incinération. Les variantes créditées sont
des BORNES (avoided burden sur une base cut-off), pas des résultats.

In [ ]:
fin_de_vie.expliquer()

In [ ]:
pd.DataFrame(fin_de_vie.bilan_systeme(ctx, jeux[REF])).T.round(1)

In [ ]:
fin_de_vie.matieres(ctx)

⚠ 12 LCA : compter deux minutes. Le tableau sert aussi à la figure.

In [ ]:
tab_eol = resultats.donnees_fin_de_vie(ctx, jeux)
resultats.figure_fin_de_vie_diptyque(ctx, jeux, modalite_detail="SH51",
                                     tableau=tab_eol);

### Hors climat : les catégories où la filière discrimine vraiment

In [ ]:
(tab_eol[(tab_eol.code == "SH51") & (tab_eol.categorie != "climate change")]
 .pivot(index="categorie", columns="scenario", values="% enfouissement")
 .round(1))

### Les sept modalités, carbone, quatre configurations

In [ ]:
pd.DataFrame({
    libelle: {fin_de_vie.LIBELLES[cle]: systeme.gwp_g(ctx, **{**v, "fin_de_vie": cle})
              for cle in fin_de_vie.MODALITES}
    for libelle, v in jeux.items()
}).round(1)

## 8. Aluminium — origine et anodisation, § 3.2

In [ ]:
structure.facteurs_aluminium(ctx)

In [ ]:
resultats.resume_aluminium(ctx, jeux[REF])

## 9. Le câble, par mètre posé — § 4.2

`intensites_carbone` ne donne que le carbone ; `intensites_cable` donne les
seize catégories. La dernière colonne extrapole la pente locale sur toute la
longueur : c'est un contrôle de linéarité, pas un chiffre à citer tel quel.

In [ ]:
resultats.intensites_carbone(ctx, jeux[REF]).round(4)

In [ ]:
tab_cable = resultats.intensites_cable(ctx, jeux[REF])
tab_cable[tab_cable.index.str.contains("toxicity|ecotoxicity")]

## 10. Lecture endpoint ReCiPe 2016 (H) — § 3.3, figure 8

⚠ Ce n'est PAS une agrégation des résultats EF : EF v3.1 n'a pas de niveau
endpoint. Les deux lectures portent sur le même inventaire et ne sont pas
convertibles l'une en l'autre.

In [ ]:
endpoint.verifier_disponibilite()   # les 3 méthodes sont-elles dans le projet ?
endpoint.verifier(ctx, jeux[REF])   # total == somme des sous-catégories ?

tab = endpoint.scores(ctx, jeux)    # les trois chiffres, 4 configurations
tab

In [ ]:
endpoint.voies(ctx, jeux[REF])        # les 22 voies, cumul à 90 %

In [ ]:
endpoint.sans_long_terme(ctx, jeux)   # (H) vs (H) no LT

In [ ]:
endpoint.figure(ctx, jeux);

## 11. CO₂ évité — § 3.4, tableau 6

In [ ]:
tab_co2 = territoire.co2_evite(ctx, jeux)
tab_co2.round(2)

In [ ]:
territoire.figure_co2_evite(tab_co2);

In [ ]:
territoire.co2_evite_scenarios(ctx, jeux, verbeux=False).round(2)

## 12. Transposition aux 54 îles — § 3.6, figures 9 et 10, annexe S2

⚠ 4 configurations × 54 îles : compter quelques minutes.

In [ ]:
iles = territoire.transposer(ctx, jeux)
(iles.sort_values("CO₂ évité net t/an", ascending=False)
     [["ile", "archipel", "dist_goelette_km", "productible_kwh_kwc_an",
       "fe_mix", "GWP g/kWh", "E nette MWh/an", "CO₂ évité net t/an",
       "Retour carbone an"]])

Table S2.3 de l'annexe, telle qu'elle y figure :

In [ ]:
(iles.sort_values("CO₂ évité net t/an", ascending=False)
     .to_csv("iles_54.csv", index=False))
print("écrit : iles_54.csv")

In [ ]:
territoire.figure_carte(iles);

In [ ]:
territoire.figure_gwp_iles(iles);
territoire.figure_co2_evite_iles(iles);

In [ ]:
territoire.figure_compromis(iles);

### Contre-vérification du productible par NASA POWER — annexe S2, sections S2.3 et S2.4

Deux sources indépendantes dans leurs entrées comme dans leur méthode : un modèle
de transposition sur données de stations interpolées d'un côté, une réanalyse
satellitaire de l'autre.

In [ ]:
nasa = territoire.comparer_nasa_power(iles=iles)
nasa[["ile", "archipel", "pvsyst_kwh_an", "ghi_nasa_kwh_m2_j",
      "ratio_pvsyst", "ratio_nasa", "ecart_%"]].round(3)

In [ ]:
territoire.figure_nasa_power(nasa);

Et la transposition entière refaite sur les ratios NASA — le test le plus dur :

In [ ]:
iles_nasa = territoire.donnees_iles(source="nasa")
tr_nasa = territoire.transposer(ctx, jeux, iles=iles_nasa, verbeux=False)
comparaison = (iles[["ile", "GWP g/kWh"]]
               .merge(tr_nasa[["ile", "GWP g/kWh"]], on="ile",
                      suffixes=(" PVsyst", " NASA")))
comparaison["écart %"] = (comparaison["GWP g/kWh NASA"]
                          / comparaison["GWP g/kWh PVsyst"] - 1) * 100
comparaison.sort_values("écart %").round(2)

## 13. Benchmark littérature harmonisé — § 2.9 et § 4.1, figure 6

In [ ]:
bench = litterature.harmoniser(ctx, jeux)
bench[["systeme", "categorie", "GWP publié g/kWh", "Productible kWh/kWc/an",
       "Durée de vie an", "GWP harmonisé g/kWh"]].round(1)

In [ ]:
litterature.figure_benchmark(bench);

In [ ]:
litterature.figure_intensite(bench);

## 14. CAPEX et LCOE — § 3.3, tableau 5

`verifier_lcoe()` d'abord : un LCOE faux passe inaperçu, il n'a pas d'ordre de
grandeur évident.

In [ ]:
economie.verifier_lcoe()

tab_eco = economie.table()                                    # base comptable
tab_eco_bis = economie.table(capex=economie.CAPEX_REFERENCE)  # base révisée

In [ ]:
economie.figure_positionnement();

## 15. Batterie de contrôles

À lancer avant toute diffusion de chiffres. Un échec ici invalide les cellules
précédentes.

In [ ]:
tests.tout(ctx, jeux, REF)

## 16. Export des figures de l'article

In [ ]:
resultats.exporter_figures(ctx, jeux, dossier="figures", dpi=300)